# 09 Feature Engineering

## 1. Objective

Objective:
Turn the EDA findings into engineered candidate features that can support later feature selection and modeling.

This notebook focuses on:
- defining feature logic from the earlier analysis
- creating transformed, grouped, interaction, and business features
- documenting redundancy and exploratory feature usefulness
- saving the engineered candidate dataset for the next stage


## Imports


In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CATEGORICAL_COLUMNS, MODEL_TARGET_COLUMN, NUMERICAL_COLUMNS, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data
from src.features.feature_engineering import (
    build_feature_selection_support,
    engineer_telco_features,
    export_engineered_dataset,
)


## 2. Load Data

Load the cleaned dataset and separate the target, numerical, and categorical feature groups.


In [94]:
df = load_cleaned_data()

target_column = TARGET_COLUMN
numerical_columns = [column for column in NUMERICAL_COLUMNS if column in df.columns]
categorical_columns = [column for column in CATEGORICAL_COLUMNS if column in df.columns]

feature_groups = {
    "target_column": target_column,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "shape": df.shape,
}

feature_groups


{'target_column': 'Churn',
 'numerical_columns': ['tenure', 'MonthlyCharges', 'TotalCharges'],
 'categorical_columns': ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod'],
 'shape': (7043, 21)}

## 3. Revisit EDA Findings

- `tenure`, `Contract`, `PaymentMethod`, `TechSupport`, `OnlineSecurity`, and `InternetService` were among the strongest churn-related signals in the earlier notebooks.
- `tenure` and `TotalCharges` were strongly related, so lifecycle redundancy needs to be documented carefully.
- High-risk combinations such as `Month-to-month + Electronic check` and `Fiber optic + No TechSupport` suggest interaction and business features are worth creating.
- This notebook therefore focuses on feature design and preview, not on fitting the final leakage-safe preprocessing pipeline.


## 4. Numerical Feature Engineering

Create numerical features that reflect the earlier findings without fitting dataset-wide preprocessing objects. At this stage, the focus is on deterministic transformations such as log features and business-friendly bins.


In [ ]:
feature_df = engineer_telco_features(df, target_column=target_column)
target_column = MODEL_TARGET_COLUMN

numerical_feature_summary = {
    "numerical_features_added": [
        column for column in ["log_total_charges", "tenure_band"] if column in feature_df.columns
    ],
    "feature_shape": feature_df.shape,
}

binary_encoded_columns = [
    column
    for column in ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling", "SeniorCitizen", target_column]
    if column in feature_df.columns
]
categorical_transform_summary = {
    "binary_encoded_in_place": binary_encoded_columns,
    "grouped_features": [
        column for column in ["payment_method_group", "internet_service_group"] if column in feature_df.columns
    ],
    "ordinal_features": [column for column in ["Contract_ordinal"] if column in feature_df.columns],
    "one_hot_encode_later": [
        column
        for column in [
            "Contract",
            "PaymentMethod",
            "InternetService",
            "tenure_band",
            "payment_method_group",
            "internet_service_group",
            "contract_payment_profile",
            "internet_techsupport_profile",
            "internet_onlinesecurity_profile",
        ]
        if column in feature_df.columns
    ],
}
interaction_feature_summary = {
    "interaction_features": [
        column
        for column in [
            "tenure_x_MonthlyCharges",
            "contract_payment_profile",
            "internet_techsupport_profile",
            "internet_onlinesecurity_profile",
        ]
        if column in feature_df.columns
    ]
}
business_feature_summary = {
    "business_features": [
        column
        for column in [
            "avg_monthly_spend",
            "is_new_customer",
            "has_support_services",
            "is_month_to_month",
            "is_auto_payment",
            "service_count",
        ]
        if column in feature_df.columns
    ]
}

numerical_feature_summary


## 5. Categorical Feature Transformations

Transform categorical features in a lightweight way by grouping and binary mapping where the business meaning is clear. Full one-hot encoding is intentionally deferred to the later modeling pipeline.


In [ ]:
categorical_transform_summary


## 6. Interaction Features

Create interaction-style features that reflect the multivariate risk combinations found earlier, using readable combined categories instead of full encoded cross-products.


In [ ]:
interaction_feature_summary


## 7. Business Features

Create business-friendly indicators that summarize customer state more directly than the raw columns alone.


In [ ]:
business_feature_summary


## 8. Redundancy Handling

The raw `TotalCharges` feature is log-transformed into `log_total_charges` to reduce right skew and then removed so the dataset keeps only one cumulative-charge representation. This makes the transformation explicit, avoids carrying both raw and transformed versions into feature selection, and reduces redundancy with other spend-related variables. `avg_monthly_spend` is still retained as a separate business feature because it captures spending intensity rather than cumulative value.

`Contract`, `Contract_ordinal`, and `is_month_to_month` are co-derived representations of the same original variable. They should not all enter the final model together. A single representation should be chosen later depending on model type, with `Contract_ordinal` generally being more suitable for tree-based models and `is_month_to_month` being a simpler option for linear models when the main risk boundary is month-to-month contracts.


## 9. Feature Selection Support (light)

Use simple exploratory ranking methods to see which engineered features look promising. This is only support analysis and should not be treated as the final leakage-safe selection step. Pearson-style correlation and mutual information are used for continuous features, while chi-square is restricted to binary, grouped categorical, and count-like engineered features.


In [ ]:
feature_selection_support_df = build_feature_selection_support(
    feature_df,
    target_column=target_column,
)

display(feature_selection_support_df.head(20))


The feature-selection support table suggests that contract-related signals remain the strongest engineered predictors, with `contract_payment_profile`, `Contract_ordinal`, and `is_month_to_month` all ranking near the top. This confirms that contract structure is central to churn risk, but it also highlights an important modeling caveat: grouped profile features such as `contract_payment_profile`, `internet_techsupport_profile`, and `internet_onlinesecurity_profile` are informative combined categories that will require careful encoding later. For cleaner baseline models, simpler features such as `Contract_ordinal`, `is_month_to_month`, `tenure`, `tenure_band`, and `is_new_customer` are easier to carry forward and interpret.


## 10. Save Feature-Engineered Dataset

Save the engineered candidate dataset and feature list as the output artifact of feature engineering. This is a handoff dataset for later stages, not the final train/test-ready modeling matrix.


In [ ]:
feature_engineering_tables_dir = project_root / "reports" / "tables" / "feature_engineering"
feature_engineering_tables_dir.mkdir(parents=True, exist_ok=True)

export_summary = export_engineered_dataset(
    feature_df,
    target_column=target_column,
    report_dir=feature_engineering_tables_dir,
)

feature_groups = {
    "target_column": target_column,
    "numerical_columns": [column for column in NUMERICAL_COLUMNS if column in df.columns],
    "categorical_columns": [column for column in CATEGORICAL_COLUMNS if column in df.columns],
    "shape": df.shape,
}
final_engineered_df = feature_df.drop(
    columns=[column for column in ["customerID"] if column in feature_df.columns]
).copy()
feature_columns = [column for column in final_engineered_df.columns if column != target_column]

pd.DataFrame([numerical_feature_summary]).to_csv(feature_engineering_tables_dir / "numerical_feature_summary.csv", index=False)
pd.DataFrame([categorical_transform_summary]).to_csv(feature_engineering_tables_dir / "categorical_transform_summary.csv", index=False)
pd.DataFrame([interaction_feature_summary]).to_csv(feature_engineering_tables_dir / "interaction_feature_summary.csv", index=False)
pd.DataFrame([business_feature_summary]).to_csv(feature_engineering_tables_dir / "business_feature_summary.csv", index=False)
feature_selection_support_df.to_csv(feature_engineering_tables_dir / "feature_selection_support.csv", index=False)

final_dataset_summary = {
    **export_summary,
    "feature_list_path": str((feature_engineering_tables_dir / "feature_list.csv").relative_to(project_root)),
    "feature_groups_path": str((feature_engineering_tables_dir / "feature_groups.csv").relative_to(project_root)),
    "numerical_feature_summary_path": str((feature_engineering_tables_dir / "numerical_feature_summary.csv").relative_to(project_root)),
    "categorical_transform_summary_path": str((feature_engineering_tables_dir / "categorical_transform_summary.csv").relative_to(project_root)),
    "interaction_feature_summary_path": str((feature_engineering_tables_dir / "interaction_feature_summary.csv").relative_to(project_root)),
    "business_feature_summary_path": str((feature_engineering_tables_dir / "business_feature_summary.csv").relative_to(project_root)),
    "feature_selection_support_path": str((feature_engineering_tables_dir / "feature_selection_support.csv").relative_to(project_root)),
    "final_dataset_summary_path": str((feature_engineering_tables_dir / "final_dataset_summary.csv").relative_to(project_root)),
}

pd.DataFrame([final_dataset_summary]).to_csv(feature_engineering_tables_dir / "final_dataset_summary.csv", index=False)

final_dataset_summary


## 11. Summary

This notebook now keeps binary features in a single, clean representation by encoding the original columns directly and converting the churn label into a numeric `target` column. It also replaces raw `TotalCharges` with a single `log_total_charges` feature so skew is handled without keeping redundant versions of the same signal. Identifier columns such as `customerID` are removed from the exported engineered dataset because they are not modeling features. Higher-cardinality or grouped categorical variables such as `Contract`, `PaymentMethod`, `InternetService`, `tenure_band`, and the profile features are intentionally left as categorical signals so they can be one-hot encoded later inside a leakage-safe modeling pipeline.
